# KG1 v146 — UMA CELL FAZ TUDO (BF16, max_seq=4096)

## Por que UMA cell so?

Apos 10 falhas em multiplas iteracoes, este notebook usa abordagem DEFENSIVA:

**BF16 puro** (NAO NF4):
- Evita TODOS os bugs Mamba+NF4 documentados
- Evita bugs LoRA+bnb_4bit em MoE experts
- Evita patches no source file

**max_seq=4096** (em vez de 8192):
- BF16 model 60GB cabe em H100 80GB
- Sobra 7-10GB para activations seq=4096
- Trunca ~50% dos samples (vs 77% com 2048)

**Trade-off de score**:
- Esperado: 0.85-0.89 (vs 0.92 ideal)
- Mas TREINA sem 10 falhas

## Setup

1. **Runtime → A100 ou H100 (HighRAM)**
2. **Colab Secrets**:
   - `HF_KEY` token (read gated repos + write repos)
3. **Aceitar termos** Nemotron: https://huggingface.co/nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16
4. **Run all** (apenas 1 cell)

Tempo: ~3-4h H100 / ~5-6h A100


In [ ]:
# CELL UNICA: setup -> deps -> mamba -> data -> model -> tokenize -> train -> save -> backup
import os, sys, time, math, gc, glob, csv, json, urllib.request, statistics, subprocess

# === ENV configs ===
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['TRANSFORMERS_NO_ADVISORY_WARNINGS'] = '1'

print('=' * 70)
print('KG1 v146 - SINGLE CELL TRAINING (BF16, max_seq=4096)')
print('=' * 70)

# ============================================================
# STEP 1: Token + GPU + secrets
# ============================================================
print('\n[1/9] Setup secrets + GPU check')
try:
    from google.colab import userdata
    hf_token = None
    for k in ['HF_KEY', 'HF_TOKEN']:
        try:
            v = userdata.get(k)
            if v:
                hf_token = v
                os.environ['HF_TOKEN'] = v
                print(f'  HF token via {k} ({len(v)} chars)')
                break
        except: pass
    if not hf_token:
        raise RuntimeError('HF_KEY missing - configure Colab Secret HF_KEY com Read gated repos')

    try:
        os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
        os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
    except: pass
except ImportError:
    raise RuntimeError('Run no Colab')

import torch
if not torch.cuda.is_available():
    raise RuntimeError('GPU required')

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'  GPU: {torch.cuda.get_device_name(0)} | VRAM: {vram_gb:.1f} GB')

if vram_gb < 70:
    print(f'  [WARN] VRAM={vram_gb:.1f}GB pode dar OOM em BF16 30B. Ideal: H100/A100 80GB')

# ============================================================
# STEP 2: Install dependencies
# ============================================================
print('\n[2/9] Install dependencies')
deps = [
    'transformers>=4.48.0', 'peft>=0.14.0', 'trl>=0.14.0', 'datasets>=3.0.0',
    'accelerate>=1.3.0', 'bitsandbytes>=0.45.0', 'huggingface_hub>=0.27.0',
    'torchao>=0.16.0', 'einops', 'packaging', 'ninja', 'triton',
]
for pkg in deps:
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', pkg],
                      capture_output=True, text=True, timeout=300)
    print(f'  {"[OK]" if r.returncode == 0 else "[FAIL]"} {pkg}')

# Force reload modules
for m in ['transformers', 'peft', 'trl', 'torchao', 'bitsandbytes']:
    if m in sys.modules:
        del sys.modules[m]

# ============================================================
# STEP 3: Install mamba-ssm v2.3.1 + causal-conv1d v1.6.1
# ============================================================
print('\n[3/9] Install mamba-ssm 2.3.1 + causal-conv1d 1.6.1')
import torch
py_ver = f'cp{sys.version_info.major}{sys.version_info.minor}'
torch_short = '.'.join(torch.__version__.split('+')[0].split('.')[:2])
abi_str = 'TRUE' if torch.compiled_with_cxx11_abi() else 'FALSE'
print(f'  Detected: {py_ver} | torch{torch_short} | abi{abi_str}')

attempts = [('cu12', torch_short, abi_str)]
for t in ['2.10', '2.9', '2.8']:
    for abi in ['TRUE', 'FALSE']:
        if (('cu12', t, abi)) not in attempts:
            attempts.append(('cu12', t, abi))

success = False
for cu, t, abi in attempts:
    cc_url = f'https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.6.1.post4/causal_conv1d-1.6.1+{cu}torch{t}cxx11abi{abi}-{py_ver}-{py_ver}-linux_x86_64.whl'
    mb_url = f'https://github.com/state-spaces/mamba/releases/download/v2.3.1/mamba_ssm-2.3.1+{cu}torch{t}cxx11abi{abi}-{py_ver}-{py_ver}-linux_x86_64.whl'
    ok_cc = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', cc_url],
                          capture_output=True, text=True, timeout=600).returncode == 0
    ok_mb = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', mb_url],
                          capture_output=True, text=True, timeout=600).returncode == 0
    if ok_cc and ok_mb:
        print(f'  [OK] mamba_ssm 2.3.1 + causal_conv1d 1.6.1 ({cu} torch{t} abi{abi})')
        success = True
        break

if not success:
    raise RuntimeError('mamba-ssm install failed')

# Force reload
for m in ['mamba_ssm', 'causal_conv1d', 'selective_scan_cuda', 'causal_conv1d_cuda']:
    if m in sys.modules:
        del sys.modules[m]

# Verify
import mamba_ssm, causal_conv1d
print(f'  mamba_ssm: {mamba_ssm.__version__}')

# ============================================================
# STEP 4: Download datasets
# ============================================================
print('\n[4/9] Download datasets (4 sources)')
DATA_DIR = '/content/data'
os.makedirs(DATA_DIR, exist_ok=True)

URLS = {
    'tong.csv': 'https://huggingface.co/datasets/felipesp1983/nemotron-cot-tong-dgxchen/resolve/main/less_cot.csv',
    'cryptarithm_813.jsonl': 'https://huggingface.co/datasets/felipesp1983/nemotron-cryptarithm-cot-813/resolve/main/cryptarithm_cot_813.jsonl',
    'eq_guess_150.jsonl': 'https://gist.githubusercontent.com/FELIPEACASTRO/0d4674009f3886d6add5be636292001a/raw',
    'synth_4425.jsonl': 'https://huggingface.co/datasets/felipesp1983/nemotron-cryptarithm-synth-4425/resolve/main/synth_cryptarithm_verified.jsonl',
}
for name, url in URLS.items():
    out = os.path.join(DATA_DIR, name)
    urllib.request.urlretrieve(url, out)
    sz = os.path.getsize(out)
    print(f'  [OK] {name}: {sz/1e6:.2f} MB' if sz > 1e6 else f'  [OK] {name}: {sz/1e3:.1f} KB')

# ============================================================
# STEP 5: Merge + Curriculum
# ============================================================
print('\n[5/9] Format + Merge + Curriculum')
from datasets import Dataset

PROMPT_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'
all_data = []

# Tong
with open(os.path.join(DATA_DIR, 'tong.csv'), encoding='utf-8') as f:
    rd = csv.DictReader(f)
    for row in rd:
        p = (row.get('prompt') or '').strip()
        c = (row.get('generated_cot') or '').strip()
        if p and c:
            all_data.append({'prompt': p + PROMPT_SUFFIX, 'completion': c, 'source': 'tong'})

# Cryptarithm 813
with open(os.path.join(DATA_DIR, 'cryptarithm_813.jsonl'), encoding='utf-8') as f:
    for line in f:
        row = json.loads(line)
        if row.get('is_valid') and row.get('cot'):
            all_data.append({
                'prompt': row['prompt'].strip() + PROMPT_SUFFIX,
                'completion': row['cot'].strip(),
                'source': 'cryptarithm_813',
            })

# EqGuess 150
with open(os.path.join(DATA_DIR, 'eq_guess_150.jsonl'), encoding='utf-8') as f:
    for line in f:
        row = json.loads(line)
        p = (row.get('prompt') or '').strip()
        c = (row.get('generated_cot') or '').strip()
        if p and c:
            all_data.append({'prompt': p + PROMPT_SUFFIX, 'completion': c, 'source': 'eq_guess_150'})

# Synth 4425
with open(os.path.join(DATA_DIR, 'synth_4425.jsonl'), encoding='utf-8') as f:
    for line in f:
        row = json.loads(line)
        p = (row.get('prompt') or '').strip()
        c = (row.get('generated_cot') or '').strip()
        if p and c:
            all_data.append({'prompt': p + PROMPT_SUFFIX, 'completion': c, 'source': 'synth_4425'})

# Curriculum
ORDER = {'tong': 0, 'eq_guess_150': 1, 'cryptarithm_813': 2, 'synth_4425': 3}
all_data.sort(key=lambda x: ORDER.get(x['source'], 99))
ds = Dataset.from_list(all_data)
print(f'  TOTAL: {len(ds)} samples (curriculum sorted)')

# ============================================================
# STEP 6: Load model BF16 (NAO NF4 - evita bugs Mamba+NF4)
# ============================================================
print(f'\n[6/9] Load Nemotron-30B em BF16 puro')
from huggingface_hub import HfApi
whoami = HfApi(token=hf_token).whoami()
print(f'  HF user: {whoami["name"]}')

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

MODEL_NAME = 'nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16'

print('  Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True, token=hf_token)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f'  Loading model 30B BF16 (~60GB - download ~3min se cache vazio)...')
t0 = time.time()
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.bfloat16,
    device_map='auto',
    trust_remote_code=True,
    token=hf_token,
    attn_implementation='eager',
)
model.enable_input_require_grads()
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
model.config.use_cache = False
print(f'  [OK] Model loaded in {time.time()-t0:.1f}s')
print(f'  VRAM after model: {torch.cuda.memory_allocated()/1e9:.1f} GB')

# Apply LoRA
TARGET_REGEX = r'.*\.(in_proj|out_proj|q_proj|k_proj|v_proj|o_proj|up_proj|down_proj|gate_proj|lm_head)$'
model = get_peft_model(model, LoraConfig(
    r=32, lora_alpha=32, target_modules=TARGET_REGEX,
    lora_dropout=0.0, bias='none', task_type='CAUSAL_LM',
))
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'  [OK] LoRA r=32 a=32: {trainable/1e6:.1f}M / {total/1e9:.2f}B ({100*trainable/total:.3f}%)')
print(f'  VRAM after LoRA: {torch.cuda.memory_allocated()/1e9:.1f} GB / {vram_gb:.1f} GB')

# ============================================================
# STEP 7: Tokenize com max_length=4096
# ============================================================
print('\n[7/9] Tokenize (max_length=4096)')
MAX_LENGTH = 4096

def fmt(ex):
    messages = [
        {'role': 'user', 'content': ex['prompt']},
        {'role': 'assistant', 'content': ex['completion']},
    ]
    full_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False, enable_thinking=True)
    prompt_text = tokenizer.apply_chat_template([messages[0]], tokenize=False, add_generation_prompt=True, enable_thinking=True)
    full_ids = tokenizer(full_text, truncation=True, max_length=MAX_LENGTH, return_tensors=None, add_special_tokens=False)['input_ids']
    prompt_ids = tokenizer(prompt_text, truncation=True, max_length=MAX_LENGTH, return_tensors=None, add_special_tokens=False)['input_ids']
    labels = list(full_ids)
    n_prompt = min(len(prompt_ids), len(labels))
    for i in range(n_prompt):
        labels[i] = -100
    return {'input_ids': full_ids, 'attention_mask': [1] * len(full_ids), 'labels': labels}

t0 = time.time()
tokenized = ds.map(fmt, remove_columns=ds.column_names, num_proc=4, desc='Tokenizing')
print(f'  [OK] Tokenized {len(tokenized)} samples in {time.time()-t0:.1f}s')

seq_lens = [len(tokenized[i]['input_ids']) for i in range(min(200, len(tokenized)))]
print(f'  Sample lengths: min={min(seq_lens)} median={statistics.median(seq_lens):.0f} max={max(seq_lens)}')

# ============================================================
# STEP 8: TRAIN com 6 OOM fixes
# ============================================================
print('\n[8/9] TRAIN (lr=5e-5, epochs=2, cosine, paged_adamw_8bit, grad_ckpt)')
gc.collect()
torch.cuda.empty_cache()

OUTPUT_DIR = '/content/output_v146'
os.makedirs(OUTPUT_DIR, exist_ok=True)

PAD_TO_MULTIPLE_OF = 8
PAD_TOKEN_ID = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id

def custom_collator(features):
    max_len = max(len(f['input_ids']) for f in features)
    max_len = ((max_len + PAD_TO_MULTIPLE_OF - 1) // PAD_TO_MULTIPLE_OF) * PAD_TO_MULTIPLE_OF
    input_ids_batch, attention_mask_batch, labels_batch = [], [], []
    for f in features:
        pad_len = max_len - len(f['input_ids'])
        input_ids_batch.append(f['input_ids'] + [PAD_TOKEN_ID] * pad_len)
        attention_mask_batch.append(f['attention_mask'] + [0] * pad_len)
        labels_batch.append(f['labels'] + [-100] * pad_len)
    return {
        'input_ids': torch.tensor(input_ids_batch, dtype=torch.long),
        'attention_mask': torch.tensor(attention_mask_batch, dtype=torch.long),
        'labels': torch.tensor(labels_batch, dtype=torch.long),
    }

from transformers import Trainer, TrainingArguments, TrainerCallback

class LossExplosionCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and 'loss' in logs:
            l = logs['loss']
            if l > 15.0 or (l != l):
                print(f'\n[WARN] Loss explosion at step {state.global_step}: {l}')
                control.should_training_stop = True

N_TRAIN = len(tokenized)
N_STEPS = math.ceil(N_TRAIN / 32) * 2
WARMUP_STEPS = int(N_STEPS * 0.03)

train_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=32,
    learning_rate=5e-5,
    lr_scheduler_type='cosine',
    warmup_steps=WARMUP_STEPS,
    adam_beta1=0.9, adam_beta2=0.95, adam_epsilon=1e-8,
    weight_decay=0.01, max_grad_norm=1.0,
    logging_steps=10, save_steps=200, save_total_limit=3,
    bf16=True,
    optim='paged_adamw_8bit',
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    remove_unused_columns=False, report_to='none',
    dataloader_num_workers=0,
)

trainer = Trainer(
    model=model, args=train_args,
    train_dataset=tokenized, data_collator=custom_collator,
    callbacks=[LossExplosionCallback()],
)
print(f'  Steps: {N_STEPS} | Warmup: {WARMUP_STEPS}')
print(f'  Time estimate: ~3-4h H100')
print()

t0 = time.time()
trainer.train()
train_time_min = (time.time() - t0) / 60
print(f'\n[OK] Training complete in {train_time_min:.1f} min')

# ============================================================
# STEP 9: Save + Upload + Backup
# ============================================================
print('\n[9/9] Save + Upload HF + GDrive backup')
ADAPTER_DIR = f'{OUTPUT_DIR}/final_adapter'
trainer.save_model(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f'  [OK] Saved at {ADAPTER_DIR}')

# Upload HF
try:
    from huggingface_hub import HfApi, create_repo
    DEST_REPO = 'felipesp1983/kg1-v146-final'
    # reload token
    try:
        from google.colab import userdata
        for k in ['HF_KEY', 'HF_TOKEN']:
            try:
                v = userdata.get(k)
                if v:
                    hf_token = v
                    break
            except: pass
    except: pass
    api = HfApi(token=hf_token)
    create_repo(DEST_REPO, private=False, exist_ok=True, token=hf_token)
    api.upload_folder(folder_path=ADAPTER_DIR, repo_id=DEST_REPO, repo_type='model',
                      commit_message=f'v146 BF16 max_seq=4096 trained {train_time_min:.1f}min')
    print(f'  [OK] HF: https://huggingface.co/{DEST_REPO}')
except Exception as e:
    print(f'  [WARN] HF upload: {e}')

# GDrive backup
try:
    import shutil
    from google.colab import drive
    drive.mount('/content/drive')
    GD = '/content/drive/My Drive/KG1_v146_adapter'
    if os.path.exists(GD):
        shutil.rmtree(GD)
    shutil.copytree(ADAPTER_DIR, GD)
    print(f'  [OK] GDrive: {GD}')
except Exception as e:
    print(f'  [WARN] GDrive: {e}')

print('\n' + '=' * 70)
print('SUCCESS - v146 training complete!')
print(f'Adapter: https://huggingface.co/felipesp1983/kg1-v146-final')
print('=' * 70)
